Study assimilation - postprocess PRO files

In [ ]:
# from edelassim.postprocess_surfex.pro import postprocess_pro
# Preprocess simulation output (assimilation of snow depth in this case)
import glob
from datetime import timedelta

import numpy as np
import xarray as xr
from pandas import date_range
from pyproj import CRS
import pandas as pd

simulation_folder = "/home/imperatoren/work/edelweiss_assimilation/simulations/edelweiss/grandesrousses250m/TEST_assim_viirs"
output_folder = "/home/imperatoren/work/edelweiss_assimilation/simulations/postprocess/grandesrousses250m/TEST_assim_viirs/"
filename = "all_members/pro/PRO.nc"
output_file = f"{output_folder}/{filename}"


def postprocess_pro(simulation_folder: str, output_file: str | None = None) -> xr.Dataset:

    member_folders = sorted(glob.glob(f"{simulation_folder}/mb*"))
    member_simulations = []
    member_numbers = []
    for member_folder in member_folders:
        member_all_period = xr.open_mfdataset(
            sorted(glob.glob(f"{member_folder}/pro/*.nc")), concat_dim="time", combine="nested"
        )
        member_all_period = member_all_period.resample(time="1d").nearest()
        member_simulations.append(member_all_period)
        member_numbers.append(int(member_folder.split("/")[-1][2:]))
    # all_edel = all_edel.assign_coords({"member": np.arange(17)})

    # date_range(all_edel.coords['time'][0])

    # all_edel_simplified = all_edel.sel(time=time_sampling)
    all_edel = xr.concat(member_simulations, dim=pd.Index(member_numbers, name="member"), coords="all")
    all_edel = all_edel.drop_vars("Projection_Type")
    all_edel = all_edel.rename({"xx": "x", "yy": "y"})
    all_edel = all_edel.rio.write_crs(CRS.from_epsg(2154)).rio.write_coordinate_system()
    if output_file is not None:
        all_edel.to_netcdf(output_file)
    return all_edel


sd_analysis = postprocess_pro(simulation_folder=simulation_folder, output_file=output_file)

Postprocess prep

In [11]:
from pyproj import CRS
import glob
from edelassim.postprocess_surfex.prep import compute_all_members_snow_tickness_and_mass
from edelassim.observation_operators import zaitchik

# from postprocess_surfex.prep import compute_snow_thickness_from_prep
slope_file = "/home/imperatoren/work/edelweiss_assimilation/data/auxiliary/topography/slope.tif"
new_prep_analysis = compute_all_members_snow_tickness_and_mass(
    simulation_folder=simulation_folder, slope_file=slope_file, type="analysis"
)
new_prep_analysis = new_prep_analysis.assign(
    {"fsc": zaitchik(swe=new_prep_analysis.data_vars["swe"], tau_scf=4, swe_full_snow_cover=200)}
)
filename = "all_members/prep/analysis/PREP_2022026_zaitchik.nc"
output_file = f"{output_folder}/{filename}"
new_prep_analysis.to_netcdf(output_file)

new_prep_background = compute_all_members_snow_tickness_and_mass(
    simulation_folder=simulation_folder, slope_file=slope_file, type="background"
)
new_prep_background = new_prep_background.assign(
    {"fsc": zaitchik(swe=new_prep_background.data_vars["swe"], tau_scf=4, swe_full_snow_cover=200)}
)
filename = "all_members/prep/background/PREP_2022026_zaitchik.nc"
output_file = f"{output_folder}/{filename}"
new_prep_background.to_netcdf(output_file)